# LeetCode 424 — Longest Repeating Character Replacement bằng Python

Link đề: https://leetcode.com/problems/longest-repeating-character-replacement/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Sliding Window, Binary Search

Skeleton LeetCode:

```python
class Solution:
    def characterReplacement(self, s: str, k: int) -> int:
        ...
```

Ghi chú: phần phát biểu đề dịch sang tiếng Việt từ đề gốc; các ví dụ đều được chạy lại bằng code trong notebook.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho chuỗi chữ in hoa `s` và số nguyên `k`; được đổi **tối đa `k`** ký tự bất kỳ thành chữ in hoa khác; tìm độ dài **lớn nhất** của đoạn con liên tiếp gồm toàn một loại chữ sau các phép đổi.
2. Nắm công cụ Python: `collections.Counter`, mảng đếm 26 phần tử với `ord(c) - ord("A")`, `max(...)`, hai con trỏ `l` và `r` với `enumerate`, tìm kiếm nhị phân trên đáp án.
3. Đi từ **vét cạn mọi đoạn con** → **tìm kiếm nhị phân độ dài + cửa sổ cố định** → **với từng chữ cái đích, cửa sổ trượt chứa tối đa `k` ký tự khác** → **cửa sổ trượt "chuẩn" với `max` đếm lại** → **cửa sổ trượt tối ưu `O(n)` không cần giảm `mx`**.
4. Hiểu mấu chốt: một đoạn dài `len` có thể làm đồng nhất bằng ≤ `k` phép đổi **khi và chỉ khi** `len - (số lần xuất hiện của ký tự phổ biến nhất) <= k`; và bài toán có tính **đơn điệu** (đoạn hợp lệ thì đoạn con của nó cũng hợp lệ).
5. Tránh các lỗi kinh điển: quên giảm đếm khi dịch `l`, nhầm `> k` với `>= k`, tính `mx` trên cả chuỗi thay vì trong cửa sổ, và "công thức tắt" `min(n, maxfreq + k)` bỏ qua điều kiện liên tiếp.

## 2. Đề bài & ví dụ

Cho chuỗi `s` chỉ gồm chữ cái **in hoa** và số nguyên `k`. Ta được chọn **một ký tự bất kỳ** của chuỗi và đổi nó thành một chữ in hoa khác; thao tác này làm **tối đa `k` lần**.

Trả về **độ dài lớn nhất** của một đoạn con (liên tiếp) chứa toàn **cùng một chữ cái** sau khi thực hiện các phép đổi trên.

**Ví dụ 1:** `s = "ABAB", k = 2` → `4`. Đổi hai chữ `A` thành `B` (hoặc ngược lại) được `"BBBB"`.

**Ví dụ 2:** `s = "AABABBA", k = 1` → `4`. Đổi chữ `A` ở giữa thành `B` ta được `"AABBBBA"`, đoạn `"BBBB"` dài 4. Có thể có cách khác cho cùng kết quả.

**Ràng buộc:** `1 <= s.length <= 10^5`; `s` chỉ gồm chữ in hoa tiếng Anh; `0 <= k <= s.length`.

In [1]:
from collections import Counter
from typing import List
import random
import time

examples = [
    ("ABAB", 2, 4),
    ("AABABBA", 1, 4),
]
for s, k, expected in examples:
    print(repr(s), "k =", k, "-> kỳ vọng", expected)

'ABAB' k = 2 -> kỳ vọng 4
'AABABBA' k = 1 -> kỳ vọng 4


### 2.1. Đọc kỹ đề: vài điểm dễ hiểu nhầm

- **Đổi ký tự trong chuỗi gốc**, nhưng kết quả chỉ quan tâm một **đoạn con** nên các phép đổi **ngoài đoạn đó là vô ích**. Vì vậy ta có thể xét từng đoạn con độc lập: *"đoạn này có thể thành toàn một chữ với ≤ `k` phép đổi không?"*
- **Đổi thành chữ nào cũng được**, nên với một đoạn ta sẽ **chọn chữ phổ biến nhất** trong đoạn làm đích để tốn ít phép đổi nhất.
- `k = 0`: không đổi gì; đáp án là độ dài **dãy ký tự giống nhau liên tiếp** dài nhất.
- `k = n`: đổi được mọi ký tự ⇒ đáp án là `n`.
- Các phép đổi **không** cần liên tiếp nhau và có thể đổi nhiều ký tự thành cùng một chữ.

In [2]:
s = "AABABBA"
for l in range(len(s)):
    for r in range(l + 1, len(s) + 1):
        if r - l == 4:
            seg = s[l:r]
            c = Counter(seg)
            top, freq = c.most_common(1)[0]
            print(f"{seg!r}: phổ biến nhất {top!r} x{freq}, số phép đổi tối thiểu = {len(seg) - freq}")

'AABA': phổ biến nhất 'A' x3, số phép đổi tối thiểu = 1
'ABAB': phổ biến nhất 'A' x2, số phép đổi tối thiểu = 2
'BABB': phổ biến nhất 'B' x3, số phép đổi tối thiểu = 1
'ABBA': phổ biến nhất 'A' x2, số phép đổi tối thiểu = 2


## 3. Kiến thức nền

### 3.1. Đoạn con (substring) và cắt lát `s[l:r]`

Đoạn con là một dãy ký tự **liên tiếp** của `s`. `s[l:r]` lấy ký tự từ chỉ số `l` đến `r - 1` (không gồm `r`); độ dài là `r - l`. Với chuỗi dài `n` có `n(n+1)/2` đoạn con không rỗng.

In [3]:
s = "AABAB"
n = len(s)
subs = [s[l:r] for l in range(n) for r in range(l + 1, n + 1)]
print(len(subs), "đoạn con (n(n+1)/2 =", n * (n + 1) // 2, ")")
print(subs[:8])

15 đoạn con (n(n+1)/2 = 15 )
['A', 'AA', 'AAB', 'AABA', 'AABAB', 'A', 'AB', 'ABA']


### 3.2. Đếm tần suất: `Counter` và mảng 26 phần tử

`Counter("AABAB")` cho `{'A': 3, 'B': 2}`. Khi bảng chữ cái cố định (26 chữ in hoa) ta cũng có thể dùng **mảng đếm** `cnt = [0] * 26` với chỉ số `ord(ch) - ord("A")`: `ord("A") = 65` nên `ord("C") - 65 = 2`. Mảng đếm nhanh hơn và dễ cập nhật khi cửa sổ trượt.

In [4]:
print(Counter("AABAB"))
cnt = [0] * 26
for ch in "AABAB":
    cnt[ord(ch) - ord("A")] += 1
print(cnt[:4], "| max =", max(cnt), "| ord('A') =", ord("A"), "ord('C') - 65 =", ord("C") - 65)

Counter({'A': 3, 'B': 2})
[3, 2, 0, 0] | max = 3 | ord('A') = 65 ord('C') - 65 = 2


### 3.3. Số phép đổi tối thiểu để đoạn thành toàn một chữ

Với một đoạn có `len` ký tự và chữ phổ biến nhất xuất hiện `maxfreq` lần: ta giữ nguyên các chữ đó và đổi **mọi ký tự còn lại** ⇒ cần `len - maxfreq` phép đổi. Chọn chữ đích khác không thể rẻ hơn (vì chữ đích khác xuất hiện không nhiều hơn `maxfreq`).

In [5]:
seg = "AABABBA"[0:5]           # "AABAB"
c = Counter(seg)
print(seg, c, "-> len =", len(seg), "maxfreq =", max(c.values()), "-> cần", len(seg) - max(c.values()), "phép đổi")
for target in "AB":
    print(f"  nếu chọn đích {target}: cần {sum(1 for ch in seg if ch != target)} phép đổi")

AABAB Counter({'A': 3, 'B': 2}) -> len = 5 maxfreq = 3 -> cần 2 phép đổi
  nếu chọn đích A: cần 2 phép đổi
  nếu chọn đích B: cần 3 phép đổi


### 3.4. Hai con trỏ `l`, `r` và cửa sổ trượt (sliding window)

Cửa sổ là đoạn `s[l..r]` (cả hai đầu mút). Mỗi bước ta **mở rộng** bằng cách tăng `r`, và khi cửa sổ vi phạm điều kiện ta **thu hẹp** bằng cách tăng `l`. Cả `l` và `r` chỉ **đi tới** nên tổng bước là `O(n)`. Khi `l` tăng, nhớ **giảm bộ đếm** của ký tự rời cửa sổ.

In [6]:
s, l = "AABAB", 0
cnt = Counter()
for r, ch in enumerate(s):
    cnt[ch] += 1
    print(f"r={r} ({ch}) cửa sổ = {s[l:r + 1]!r} đếm = {dict(cnt)}")
cnt[s[l]] -= 1           # ký tự rời cửa sổ khi l tăng
l += 1
print("sau khi l -> 1:", s[l:], dict(cnt))

r=0 (A) cửa sổ = 'A' đếm = {'A': 1}
r=1 (A) cửa sổ = 'AA' đếm = {'A': 2}
r=2 (B) cửa sổ = 'AAB' đếm = {'A': 2, 'B': 1}
r=3 (A) cửa sổ = 'AABA' đếm = {'A': 3, 'B': 1}
r=4 (B) cửa sổ = 'AABAB' đếm = {'A': 3, 'B': 2}
sau khi l -> 1: ABAB {'A': 2, 'B': 2}


### 3.5. Tìm kiếm nhị phân trên đáp án và tính đơn điệu

Nếu một đoạn dài `L` làm được với ≤ `k` phép đổi thì **đoạn con của nó** (dài `L - 1`) cũng làm được (bỏ bớt một đầu không làm tăng số phép đổi cần). Vì "khả thi" là đơn điệu theo độ dài (`True ... True False ... False`), ta nhị phân để tìm độ dài lớn nhất còn khả thi. Cấu trúc: `lo, hi = 1, n`; `mid = (lo + hi + 1) // 2` (làm tròn lên); nếu khả thi `lo = mid` ngược lại `hi = mid - 1`.

In [7]:
feasible = [True, True, True, True, False, False, False]      # độ dài 1..7, ví dụ đơn điệu
lo, hi = 0, len(feasible) - 1
while lo < hi:
    mid = (lo + hi + 1) // 2
    print(f"lo={lo} hi={hi} mid={mid} feasible={feasible[mid]}")
    if feasible[mid]:
        lo = mid
    else:
        hi = mid - 1
print("độ dài lớn nhất khả thi =", lo + 1)

lo=0 hi=6 mid=3 feasible=True
lo=3 hi=6 mid=5 feasible=False
lo=3 hi=4 mid=4 feasible=False
độ dài lớn nhất khả thi = 4


## 4. Ý tưởng nền tảng: tính tay một ví dụ

Lấy `s = "AABABBA"`, `k = 1` (ví dụ 2). Ta thử vài đoạn con và hỏi: tối thiểu bao nhiêu phép đổi? Công thức: `len - maxfreq`.

| đoạn | len | maxfreq | phép đổi | ≤ k = 1? |
|---|---|---|---|---|
| `AAB` | 3 | 2 (`A`) | 1 | có |
| `AABA` | 4 | 3 (`A`) | 1 | có |
| `AABAB` | 5 | 3 (`A`) | 2 | không |
| `ABBA` | 4 | 2 | 2 | không |
| `BABB` | 4 | 3 (`B`) | 1 | có |
| `ABABB` | 5 | 3 (`B`) | 2 | không |

Các đoạn dài 4 như `AABA`, `BABB` đạt được; đoạn dài 5 nào cũng cần ≥ 2 phép đổi. Vậy đáp án là `4`. Hãy để máy làm hết các đoạn.

In [8]:
s, k = "AABABBA", 1
best = 0
for l in range(len(s)):
    for r in range(l + 1, len(s) + 1):
        seg = s[l:r]
        need = len(seg) - max(Counter(seg).values())
        if need <= k:
            best = max(best, len(seg))
print("đáp án =", best)

đáp án = 4


## 5. Cách 1 — Brute Force: thử mọi đoạn con

**Ý tưởng:** với mỗi điểm đầu `l`, mở rộng `r` sang phải, cập nhật bộ đếm và `maxfreq` của đoạn `s[l..r]`; nếu `len - maxfreq <= k` thì cập nhật đáp án. Mỗi đoạn tốn `O(1)` nhờ cập nhật đếm tăng dần (không đếm lại từ đầu).

In [9]:
def replace_brute(s, k):
    n = len(s)
    best = 0
    for l in range(n):
        cnt = [0] * 26
        maxfreq = 0
        for r in range(l, n):
            idx = ord(s[r]) - ord("A")
            cnt[idx] += 1
            maxfreq = max(maxfreq, cnt[idx])
            if (r - l + 1) - maxfreq <= k:
                best = max(best, r - l + 1)
    return best


for s, k, expected in examples:
    got = replace_brute(s, k)
    print(repr(s), k, "->", got, "| kỳ vọng", expected)
    assert got == expected

'ABAB' 2 -> 4 | kỳ vọng 4
'AABABBA' 1 -> 4 | kỳ vọng 4


**Độ phức tạp:** hai vòng lặp `l`, `r` ⇒ **Time `O(n²)`**, Space `O(26) = O(1)`.

**Nút thắt:** `n = 10^5` ⇒ `~5 * 10^9` đoạn con. Ta đã bỏ qua việc **hai đoạn liền kề chia sẻ gần như hoàn toàn nội dung**. Đo thử để thấy:

In [10]:
rng = random.Random(1)
s = "".join(rng.choice("ABCDEFGHIJKLMNOPQRSTUVWXYZ") for _ in range(3000))
t = time.perf_counter()
replace_brute(s, 10)
print("brute với n = 3000: %.2f giây (n = 10^5 sẽ chậm ~1000 lần)" % (time.perf_counter() - t))

brute với n = 3000: 0.48 giây (n = 10^5 sẽ chậm ~1000 lần)


## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát A — điều kiện hợp lệ của một đoạn:** `len - maxfreq <= k`.

**Quan sát B — tính đơn điệu:** đoạn hợp lệ thì mọi đoạn con của nó cũng hợp lệ. Điều này cho phép:

- **Nhị phân** trên độ dài đáp án (Cách 2), hoặc
- **Cửa sổ trượt** hai con trỏ: mở rộng `r`; nếu cửa sổ không hợp lệ, thu hẹp `l` (Cách 4).

**Quan sát C — mẹo "không cần giảm `mx`":** ta chỉ cần **độ dài lớn nhất**. Giữ một biến `mx` = *số lần xuất hiện lớn nhất của một chữ từng thấy trong mọi cửa sổ trước đó* (không giảm khi `l` dịch). Cửa sổ chỉ cần **dài hơn** kỷ lục hiện tại mới có ích, và muốn dài hơn thì `mx` thật sự phải tăng. Vì thế `mx` "cũ" chỉ làm điều kiện chặt hơn mức cần thiết (không bao giờ nới), ta vẫn không bỏ sót đáp án và không nhận nhầm đáp án lớn hơn thực tế. Mục 10 chứng minh kỹ.

Ta kiểm chứng điều kiện A trên các đoạn bằng mã và đối chiếu với định nghĩa "thử mọi chữ đích".

In [11]:
def need_by_definition(seg):
    # thử mọi chữ đích A..Z, đếm số ký tự khác đích -> số phép đổi tối thiểu
    return min(sum(1 for ch in seg if ch != t) for t in "ABCDEFGHIJKLMNOPQRSTUVWXYZ")


rng = random.Random(6)
for _ in range(500):
    seg = "".join(rng.choice("ABC") for _ in range(rng.randint(1, 12)))
    assert need_by_definition(seg) == len(seg) - max(Counter(seg).values())
print("OK: len - maxfreq = số phép đổi tối thiểu theo định nghĩa, trên 500 đoạn")

OK: len - maxfreq = số phép đổi tối thiểu theo định nghĩa, trên 500 đoạn


## 7. Các cách tiếp theo

### Cách 2 — Nhị phân độ dài + kiểm tra cửa sổ cố định (`O(26 n log n)`)

**Giải thích từng đoạn:**

1. `feasible(L)`: có cửa sổ độ dài `L` nào thỏa `L - maxfreq <= k` không? Trượt cửa sổ cố định `L`, giữ mảng đếm 26 phần tử; mỗi bước tính `max(cnt)` (26 phép).
2. Nhị phân `lo = 1` (luôn khả thi vì `k >= 0` và một ký tự là đồng nhất), `hi = n`.
3. `mid = (lo + hi + 1) // 2`; nếu `feasible(mid)` thì `lo = mid` ngược lại `hi = mid - 1`.

In [12]:
def replace_binary(s, k):
    n = len(s)
    codes = [ord(c) - ord("A") for c in s]

    def feasible(L):
        cnt = [0] * 26
        for i in range(n):
            cnt[codes[i]] += 1
            if i >= L:
                cnt[codes[i - L]] -= 1       # ký tự rời cửa sổ cố định
            if i >= L - 1 and L - max(cnt) <= k:
                return True
        return False

    lo, hi = 1, n
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if feasible(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo


for s, k, expected in examples:
    got = replace_binary(s, k)
    print(got, "| kỳ vọng", expected)
    assert got == expected

4 | kỳ vọng 4
4 | kỳ vọng 4


In [13]:
# Code nộp LeetCode — Cách 2
class Solution2:
    def characterReplacement(self, s: str, k: int) -> int:
        n = len(s)
        codes = [ord(c) - 65 for c in s]

        def feasible(L: int) -> bool:
            cnt = [0] * 26
            for i in range(n):
                cnt[codes[i]] += 1
                if i >= L:
                    cnt[codes[i - L]] -= 1
                if i >= L - 1 and L - max(cnt) <= k:
                    return True
            return False

        lo, hi = 1, n
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if feasible(mid):
                lo = mid
            else:
                hi = mid - 1
        return lo


print(Solution2().characterReplacement("AABABBA", 1))   # 4

4


**Độ phức tạp:** `log2(n)` vòng nhị phân, mỗi vòng quét `n` vị trí và `max` 26 phần tử ⇒ **Time `O(26 n log n)`**, Space `O(n + 26)` (mảng `codes`).

### Cách 3 — Với từng chữ cái đích: cửa sổ chứa tối đa `k` ký tự "khác đích"

**Ý tưởng khác:** cố định chữ đích `c` (26 lựa chọn). Đoạn hợp lệ cho `c` là đoạn có **không quá `k` ký tự khác `c`**. Tìm đoạn dài nhất như vậy bằng hai con trỏ (giống bài "Max Consecutive Ones III"). Đáp án là max theo mọi `c`.

**Giải thích từng đoạn:**

1. Với `c` trong các chữ cái **xuất hiện** trong `s` (chữ không xuất hiện không giúp được gì hơn).
2. `bad` = số ký tự khác `c` trong cửa sổ `s[l..r]`.
3. Mỗi bước `r`: nếu `s[r] != c` thì `bad += 1`; trong khi `bad > k`: nếu `s[l] != c` thì `bad -= 1`; `l += 1`.
4. Cập nhật `best = max(best, r - l + 1)`.

In [14]:
def replace_per_letter(s, k):
    best = 0
    for c in set(s):
        l = bad = 0
        for r, ch in enumerate(s):
            if ch != c:
                bad += 1
            while bad > k:
                if s[l] != c:
                    bad -= 1
                l += 1
            best = max(best, r - l + 1)
    return best


for s, k, expected in examples:
    got = replace_per_letter(s, k)
    print(got, "| kỳ vọng", expected)
    assert got == expected

4 | kỳ vọng 4
4 | kỳ vọng 4


In [15]:
# Code nộp LeetCode — Cách 3
class Solution3:
    def characterReplacement(self, s: str, k: int) -> int:
        best = 0
        for c in set(s):
            l = bad = 0
            for r, ch in enumerate(s):
                if ch != c:
                    bad += 1
                while bad > k:
                    if s[l] != c:
                        bad -= 1
                    l += 1
                best = max(best, r - l + 1)
        return best


print(Solution3().characterReplacement("ABAB", 2))   # 4

4


**Độ phức tạp:** tối đa 26 chữ đích, mỗi chữ hai con trỏ `O(n)` ⇒ **Time `O(26 n)`**, Space `O(26)`.

### Cách 4 — Cửa sổ trượt "chuẩn": thu hẹp bằng `while`, dùng `max` thật mỗi lần

**Giải thích từng đoạn:**

1. Giữ bộ đếm `cnt` của **cửa sổ hiện tại** `s[l..r]`.
2. Mỗi bước `r`: `cnt[s[r]] += 1`.
3. **Trong khi** `(r - l + 1) - max(cnt.values()) > k`: gỡ `s[l]` (`cnt[s[l]] -= 1`) rồi `l += 1`.
4. Cửa sổ lúc này hợp lệ; `best = max(best, r - l + 1)`.

Đây là phiên bản "đúng theo định nghĩa", tốn `max` 26 phần tử mỗi lần kiểm tra.

In [16]:
def replace_window_exact(s, k):
    cnt = [0] * 26
    l = best = 0
    for r, ch in enumerate(s):
        cnt[ord(ch) - 65] += 1
        while (r - l + 1) - max(cnt) > k:
            cnt[ord(s[l]) - 65] -= 1
            l += 1
        best = max(best, r - l + 1)
    return best


for s, k, expected in examples:
    got = replace_window_exact(s, k)
    print(got, "| kỳ vọng", expected)
    assert got == expected

4 | kỳ vọng 4
4 | kỳ vọng 4


In [17]:
# Code nộp LeetCode — Cách 4
class Solution4:
    def characterReplacement(self, s: str, k: int) -> int:
        cnt = [0] * 26
        l = best = 0
        for r, ch in enumerate(s):
            cnt[ord(ch) - 65] += 1
            while (r - l + 1) - max(cnt) > k:
                cnt[ord(s[l]) - 65] -= 1
                l += 1
            best = max(best, r - l + 1)
        return best


print(Solution4().characterReplacement("AABABBA", 1))   # 4

4


**Độ phức tạp:** `l` và `r` chỉ tiến ⇒ tổng `2n` bước; mỗi bước kiểm tra `max(cnt)` tốn 26 ⇒ **Time `O(26 n)`**, Space `O(26)`.

### Cách 5 — Cửa sổ trượt tối ưu: `mx` không giảm, cửa sổ chỉ "trượt" hoặc "lớn thêm" (`O(n)`)

**Ý tưởng:** theo Quan sát C, ta thay `max(cnt)` bằng biến `mx` chỉ **tăng** (cập nhật `mx = max(mx, cnt[s[r]])` khi thêm ký tự). Khi `r - l + 1 - mx > k` ta chỉ dịch `l` **một** bước (dùng `if`, không cần `while`) — cửa sổ khi đó **trượt** mà không to thêm. Cuối cùng cửa sổ lớn nhất chính là độ dài `n - l` (cửa sổ không bao giờ nhỏ đi).

**Giải thích từng đoạn:**

1. `cnt`, `l = mx = 0`.
2. Với mỗi `r`: `cnt[s[r]] += 1`; `mx = max(mx, cnt[s[r]])`.
3. Nếu `r - l + 1 - mx > k`: `cnt[s[l]] -= 1`; `l += 1`.
4. Đáp án: `n - l`.

In [18]:
def replace_window_fast(s, k):
    cnt = [0] * 26
    l = mx = 0
    for r, ch in enumerate(s):
        i = ord(ch) - 65
        cnt[i] += 1
        mx = max(mx, cnt[i])                 # chỉ tăng, không giảm khi l dịch
        if r - l + 1 - mx > k:               # cửa sổ vượt quá: trượt (không dùng while)
            cnt[ord(s[l]) - 65] -= 1
            l += 1
    return len(s) - l


for s, k, expected in examples:
    got = replace_window_fast(s, k)
    print(got, "| kỳ vọng", expected)
    assert got == expected

4 | kỳ vọng 4
4 | kỳ vọng 4


In [19]:
# Code nộp LeetCode — Cách 5 (khuyến nghị)
class Solution5:
    def characterReplacement(self, s: str, k: int) -> int:
        cnt = [0] * 26
        l = mx = 0
        for r, ch in enumerate(s):
            i = ord(ch) - 65
            cnt[i] += 1
            mx = max(mx, cnt[i])
            if r - l + 1 - mx > k:
                cnt[ord(s[l]) - 65] -= 1
                l += 1
        return len(s) - l


print(Solution5().characterReplacement("ABAB", 2))   # 4

4


**Độ phức tạp:** một lượt `O(n)`, mỗi bước `O(1)` ⇒ **Time `O(n)`**, Space `O(26) = O(1)`.

**Năm cách tóm tắt:** (1) vét cạn đoạn con; (2) nhị phân độ dài; (3) với từng chữ đích đếm "khác đích"; (4) cửa sổ trượt đúng định nghĩa; (5) cửa sổ trượt với `mx` không giảm. Ba cách cuối đều dựa trên tính đơn điệu nhưng khác về cách kiểm tra điều kiện.

## 8. TRACE từng bước

Trace **Cách 4** (cửa sổ chuẩn) và **Cách 5** (tối ưu) trên cùng ví dụ `s = "AABABBA"`, `k = 1`.

Bảng trạng thái của Cách 4 (dự kiến; hãy tự điền trước khi chạy):

| r | ký tự | đếm trong cửa sổ | `len - max` | hành động | cửa sổ | best |
|---|---|---|---|---|---|---|
| 0 | A | A1 | 0 | giữ | `A` | 1 |
| 1 | A | A2 | 0 | giữ | `AA` | 2 |
| 2 | B | A2 B1 | 1 | giữ | `AAB` | 3 |
| 3 | A | A3 B1 | 1 | giữ | `AABA` | 4 |
| 4 | B | A3 B2 | 2 | `l -> 1` | `ABAB` | 4 |
| ... | | | | | | |

In [20]:
def trace_exact(s, k):
    cnt = [0] * 26
    l = best = 0
    for r, ch in enumerate(s):
        cnt[ord(ch) - 65] += 1
        print(f"r={r} thêm {ch!r}: cửa sổ [{l},{r}] = {s[l:r + 1]!r}, max={max(cnt)}, len-max={r - l + 1 - max(cnt)}")
        while (r - l + 1) - max(cnt) > k:
            print(f"      len-max > {k}: bỏ {s[l]!r} ở l={l}")
            cnt[ord(s[l]) - 65] -= 1
            l += 1
            print(f"      cửa sổ mới [{l},{r}] = {s[l:r + 1]!r}")
        best = max(best, r - l + 1)
        print(f"      best = {best}")
    return best


assert trace_exact("AABABBA", 1) == 4

r=0 thêm 'A': cửa sổ [0,0] = 'A', max=1, len-max=0
      best = 1
r=1 thêm 'A': cửa sổ [0,1] = 'AA', max=2, len-max=0
      best = 2
r=2 thêm 'B': cửa sổ [0,2] = 'AAB', max=2, len-max=1
      best = 3
r=3 thêm 'A': cửa sổ [0,3] = 'AABA', max=3, len-max=1
      best = 4
r=4 thêm 'B': cửa sổ [0,4] = 'AABAB', max=3, len-max=2
      len-max > 1: bỏ 'A' ở l=0
      cửa sổ mới [1,4] = 'ABAB'
      len-max > 1: bỏ 'A' ở l=1
      cửa sổ mới [2,4] = 'BAB'
      best = 4
r=5 thêm 'B': cửa sổ [2,5] = 'BABB', max=3, len-max=1
      best = 4
r=6 thêm 'A': cửa sổ [2,6] = 'BABBA', max=3, len-max=2
      len-max > 1: bỏ 'B' ở l=2
      cửa sổ mới [3,6] = 'ABBA'
      len-max > 1: bỏ 'A' ở l=3
      cửa sổ mới [4,6] = 'BBA'
      best = 4


Cách 5 nhìn gần như vậy nhưng `mx` chỉ tăng. Hàm trace dưới đây in **cả `mx` giả (không giảm) lẫn `max` thật của cửa sổ** để thấy chỗ `mx` "cũ" và vì sao đáp án vẫn đúng: chỉ khi `mx` tăng thật sự cửa sổ mới to thêm.

In [21]:
def trace_fast(s, k):
    cnt = [0] * 26
    l = mx = 0
    for r, ch in enumerate(s):
        i = ord(ch) - 65
        cnt[i] += 1
        mx = max(mx, cnt[i])
        size = r - l + 1
        shrink = size - mx > k
        print(f"r={r} {ch!r}: mx(giả)={mx}, max thật={max(cnt)}, size={size}, size-mx={size - mx} -> {'TRƯỢT l' if shrink else 'giữ (cửa sổ to thêm)'}")
        if shrink:
            cnt[ord(s[l]) - 65] -= 1
            l += 1
        print(f"      cửa sổ = [{l},{r}] = {s[l:r + 1]!r}, kích thước = {r - l + 1}")
    print("đáp án = n - l =", len(s) - l)
    return len(s) - l


assert trace_fast("AABABBA", 1) == 4

r=0 'A': mx(giả)=1, max thật=1, size=1, size-mx=0 -> giữ (cửa sổ to thêm)
      cửa sổ = [0,0] = 'A', kích thước = 1
r=1 'A': mx(giả)=2, max thật=2, size=2, size-mx=0 -> giữ (cửa sổ to thêm)
      cửa sổ = [0,1] = 'AA', kích thước = 2
r=2 'B': mx(giả)=2, max thật=2, size=3, size-mx=1 -> giữ (cửa sổ to thêm)
      cửa sổ = [0,2] = 'AAB', kích thước = 3
r=3 'A': mx(giả)=3, max thật=3, size=4, size-mx=1 -> giữ (cửa sổ to thêm)
      cửa sổ = [0,3] = 'AABA', kích thước = 4
r=4 'B': mx(giả)=3, max thật=3, size=5, size-mx=2 -> TRƯỢT l
      cửa sổ = [1,4] = 'ABAB', kích thước = 4
r=5 'B': mx(giả)=3, max thật=3, size=5, size-mx=2 -> TRƯỢT l
      cửa sổ = [2,5] = 'BABB', kích thước = 4
r=6 'A': mx(giả)=3, max thật=3, size=5, size-mx=2 -> TRƯỢT l
      cửa sổ = [3,6] = 'ABBA', kích thước = 4
đáp án = n - l = 4


Quan sát: từ `r = 4` trở đi, `mx(giả) = 3` (kỷ lục từ cửa sổ `AABA`), trong khi cửa sổ thật sau khi trượt (`ABAB`, `BABB`, `ABBA`) chỉ có tối đa 2 chữ giống nhau. Cửa sổ vì thế chỉ **trượt** với kích thước giữ nguyên `4 = mx + k`, và đáp án `4` đúng bằng kỷ lục. Cách 5 **không bao giờ báo đáp án lớn hơn đúng**, vì để cửa sổ to lên `5` cần `mx` thật sự đạt `4`.

Cuối cùng, trace **Cách 2** (nhị phân): in từng `mid` và kết quả `feasible`.

In [22]:
def trace_binary(s, k):
    n = len(s)
    codes = [ord(c) - 65 for c in s]

    def feasible(L):
        cnt = [0] * 26
        for i in range(n):
            cnt[codes[i]] += 1
            if i >= L:
                cnt[codes[i - L]] -= 1
            if i >= L - 1 and L - max(cnt) <= k:
                return True, s[i - L + 1:i + 1]
        return False, None

    lo, hi = 1, n
    while lo < hi:
        mid = (lo + hi + 1) // 2
        ok, window = feasible(mid)
        print(f"lo={lo} hi={hi} mid={mid}: khả thi = {ok}" + (f" (ví dụ cửa sổ {window!r})" if ok else ""))
        if ok:
            lo = mid
        else:
            hi = mid - 1
    print("đáp án =", lo)
    return lo


assert trace_binary("AABABBA", 1) == 4

lo=1 hi=7 mid=4: khả thi = True (ví dụ cửa sổ 'AABA')
lo=4 hi=7 mid=6: khả thi = False
lo=4 hi=5 mid=5: khả thi = False
đáp án = 4


## 9. Code nộp LeetCode

Bản khuyến nghị: **Cách 5** — một lượt, `O(n)` thời gian, `O(1)` bộ nhớ, code ngắn. Dưới đây là bản có chú thích.

In [23]:
class Solution:
    def characterReplacement(self, s: str, k: int) -> int:
        cnt = [0] * 26              # đếm chữ trong cửa sổ hiện tại [l, r]
        l = 0
        mx = 0                      # số lần lớn nhất của một chữ từng thấy trong một cửa sổ (không giảm)
        for r, ch in enumerate(s):
            i = ord(ch) - ord("A")
            cnt[i] += 1
            mx = max(mx, cnt[i])
            if r - l + 1 - mx > k:  # cần đổi nhiều hơn k ký tự: dịch l một bước (cửa sổ trượt)
                cnt[ord(s[l]) - ord("A")] -= 1
                l += 1
        return len(s) - l           # cửa sổ không bao giờ co lại, độ dài cuối cùng = đáp án


for s, k, expected in examples:
    got = Solution().characterReplacement(s, k)
    print(got, expected)
    assert got == expected

4 4
4 4


## 10. Vì sao đúng

### Điều kiện hợp lệ của đoạn

Một đoạn có thể đồng nhất bằng ≤ `k` phép đổi ⇔ `len - maxfreq <= k`. (Giữ chữ đích là chữ phổ biến nhất; chữ đích khác chỉ tệ hơn. Xem kiểm chứng bằng định nghĩa ở mục 6.)

### Tính đơn điệu

Nếu đoạn `[a, b]` hợp lệ thì mọi đoạn con `[a', b'] ⊆ [a, b]` hợp lệ: bỏ bớt một ký tự ở đầu làm `len` giảm 1 còn `maxfreq` giảm **nhiều nhất 1**, nên `len - maxfreq` không tăng.

### Cách 5 (`mx` không giảm) — lập luận

Gọi `W_r` là kích thước cửa sổ sau bước `r` (cửa sổ `[l, r]`), `mx_r` là giá trị `mx` sau bước `r`.

**Bổ đề 1 — công thức của cửa sổ:** `W_r = min(r + 1, mx_r + k)`.

*Chứng minh bằng quy nạp.* Ở bước `r`, cửa sổ tạm thời to thêm 1 (thành `W_{r-1} + 1`). Nó chỉ phải trượt khi `W_{r-1} + 1 - mx_r > k`, tức `W_{r-1} + 1 > mx_r + k`. Do `mx` không giảm và theo giả thiết quy nạp `W_{r-1} <= mx_{r-1} + k <= mx_r + k`, nên: nếu `W_{r-1} < mx_r + k` cửa sổ to thêm (`W_r = W_{r-1} + 1`), nếu `W_{r-1} = mx_r + k` thì cửa sổ giữ nguyên kích thước. Cả hai trường hợp cho `W_r = min(r + 1, mx_r + k)`. ∎

**Bổ đề 2 — không báo quá lớn (tính an toàn):** `mx_r = M` luôn đạt được tại một cửa sổ thật `[l', r']` chứa đúng `M` chữ giống nhau. Mở rộng cửa sổ đó (thêm các ký tự lân cận trong chuỗi) tới độ dài `min(n, M + k)` ta được một đoạn có `len - maxfreq <= (M + k) - M = k`, tức hợp lệ. Vậy đáp án thuật toán `min(n, M + k)` luôn là độ dài của một đoạn **hợp lệ thật** (không bao giờ lớn hơn đáp án đúng).

**Tính đầy đủ (không bỏ sót):** cửa sổ của Cách 5 có đúng kích thước mà Cách 4 (đúng theo định nghĩa) đạt được; ta kiểm chứng điều này bằng bất biến ở ô code dưới (kích thước cửa sổ Cách 5 **bằng** độ dài hợp lệ lớn nhất tới thời điểm `r` do Cách 4 tính) trên 2000 chuỗi ngẫu nhiên, và bằng bộ test mục 12 so với oracle theo định nghĩa.

**Phiên bản dễ chứng minh hơn:** Cách 4 (dùng `max(cnt)` thật và `while`) đúng ngay theo định nghĩa: sau mỗi bước `r`, `l` là điểm đầu **nhỏ nhất** sao cho `[l, r]` hợp lệ (nhờ đơn điệu, ta có thể tăng `l` cho tới khi hợp lệ và không thể bỏ sót đoạn dài hơn kết thúc ở `r`); `best` lấy max theo mọi `r`.

Để "thấy" vì sao `mx` cũ không gây sai, ta kiểm tra bất biến bằng code: ở mọi bước, kích thước cửa sổ của Cách 5 bằng **độ dài hợp lệ lớn nhất của các đoạn kết thúc tại `r` hoặc sớm hơn** (tính bằng Cách 4).

In [24]:
def check_invariant(s, k):
    cnt = [0] * 26
    l = mx = 0
    best_exact = 0
    cnt2 = [0] * 26
    l2 = 0
    for r, ch in enumerate(s):
        i = ord(ch) - 65
        # Cách 5
        cnt[i] += 1
        mx = max(mx, cnt[i])
        if r - l + 1 - mx > k:
            cnt[ord(s[l]) - 65] -= 1
            l += 1
        # Cách 4
        cnt2[i] += 1
        while (r - l2 + 1) - max(cnt2) > k:
            cnt2[ord(s[l2]) - 65] -= 1
            l2 += 1
        best_exact = max(best_exact, r - l2 + 1)
        assert r - l + 1 == best_exact, (s, k, r)


rng = random.Random(10)
for _ in range(2000):
    s = "".join(rng.choice("ABC") for _ in range(rng.randint(1, 15)))
    check_invariant(s, rng.randint(0, len(s)))
print("OK: sau mỗi bước, kích thước cửa sổ Cách 5 = độ dài hợp lệ lớn nhất tới thời điểm đó")

OK: sau mỗi bước, kích thước cửa sổ Cách 5 = độ dài hợp lệ lớn nhất tới thời điểm đó


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn đoạn con | `O(n²)` | `O(1)` |
| 2. Nhị phân độ dài | `O(26 n log n)` | `O(n)` |
| 3. Từng chữ đích | `O(26 n)` | `O(1)` |
| 4. Cửa sổ chuẩn (`max` thật) | `O(26 n)` | `O(1)` |
| 5. Cửa sổ tối ưu (`mx` không giảm) | `O(n)` | `O(1)` |

Hãy đo trên `n = 10^5` (Cách 1 quá chậm nên bỏ qua).

In [25]:
rng = random.Random(2)
for label, alphabet, k in [("ngẫu nhiên 26 chữ, k=1000", "ABCDEFGHIJKLMNOPQRSTUVWXYZ", 1000),
                           ("chỉ AB, k=10", "AB", 10)]:
    s = "".join(rng.choice(alphabet) for _ in range(10**5))
    print(label)
    for name, f in [("Cách 2", replace_binary), ("Cách 3", replace_per_letter),
                    ("Cách 4", replace_window_exact), ("Cách 5", replace_window_fast)]:
        t = time.perf_counter()
        r = f(s, k)
        print(f"   {name}: {r} trong {time.perf_counter() - t:.3f} giây")

ngẫu nhiên 26 chữ, k=1000


   Cách 2: 1068 trong 0.354 giây


   Cách 3: 1068 trong 0.416 giây
   Cách 4: 1068 trong 0.070 giây
   Cách 5: 1068 trong 0.016 giây
chỉ AB, k=10


   Cách 2: 48 trong 0.481 giây
   Cách 3: 48 trong 0.039 giây
   Cách 4: 48 trong 0.115 giây
   Cách 5: 48 trong 0.019 giây


## 12. Bộ test

Oracle độc lập: `replace_definition(s, k)` thử **mọi đoạn con** và **mọi chữ đích A..Z**, đếm số ký tự khác đích; đoạn hợp lệ nếu có chữ đích với số ký tự khác ≤ `k`. Nó không dùng công thức `len - maxfreq` nào.

In [26]:
def replace_definition(s, k):
    best = 0
    n = len(s)
    for l in range(n):
        for r in range(l + 1, n + 1):
            seg = s[l:r]
            ok = any(sum(1 for ch in seg if ch != t) <= k for t in "ABCDEFGHIJKLMNOPQRSTUVWXYZ")
            if ok:
                best = max(best, r - l)
    return best


solutions = {
    "brute": replace_brute,
    "binary": replace_binary,
    "per-letter": replace_per_letter,
    "window-exact": replace_window_exact,
    "window-fast": replace_window_fast,
    "LeetCode Solution": lambda s, k: Solution().characterReplacement(s, k),
    "Solution2": lambda s, k: Solution2().characterReplacement(s, k),
    "Solution3": lambda s, k: Solution3().characterReplacement(s, k),
    "Solution4": lambda s, k: Solution4().characterReplacement(s, k),
    "Solution5": lambda s, k: Solution5().characterReplacement(s, k),
}

# 12.1. Ví dụ đề
for s, k, expected in examples:
    assert replace_definition(s, k) == expected
    for name, f in solutions.items():
        assert f(s, k) == expected, (name, s, k)
print("12.1 OK")

12.1 OK


In [27]:
# 12.2. Edge case
edge = [
    ("A", 0, 1),
    ("A", 1, 1),
    ("AB", 0, 1),
    ("AB", 1, 2),
    ("AAAA", 0, 4),
    ("ABCDE", 0, 1),                 # k = 0: dãy giống nhau dài nhất
    ("ABCDE", 4, 5),                 # k = n - 1
    ("ABCDE", 5, 5),                 # k = n
    ("AABBAABB", 0, 2),
    ("ABABAB", 1, 3),                # công thức tắt min(n, maxfreq + k) = 4 sẽ sai
    ("ABABAB", 2, 5),
    ("AAAAABBBBB", 1, 6),
    ("BAAAB", 1, 4),                 # đổi một B thành A: AAAA dài 4
]
for s, k, expected in edge:
    truth = replace_definition(s, k)
    for name, f in solutions.items():
        got = f(s, k)
        assert got == truth, (name, s, k, got, truth)
print("12.2 OK:", len(edge), "edge case; ABABAB k=1 ->", replace_definition("ABABAB", 1))

12.2 OK: 13 edge case; ABABAB k=1 -> 3


In [28]:
# 12.3. Ngẫu nhiên có seed, đối chiếu với oracle theo định nghĩa
rng = random.Random(424)
for _ in range(1500):
    n = rng.randint(1, 14)
    alphabet = rng.choice(["AB", "ABC", "ABCD", "ABCDEFGHIJKLMNOPQRSTUVWXYZ"])
    s = "".join(rng.choice(alphabet) for _ in range(n))
    k = rng.randint(0, n)
    truth = replace_definition(s, k)
    for name, f in solutions.items():
        got = f(s, k)
        assert got == truth, (name, s, k, got, truth)
print("12.3 OK: 1500 test ngẫu nhiên khớp oracle theo định nghĩa ở cả 10 cách")

12.3 OK: 1500 test ngẫu nhiên khớp oracle theo định nghĩa ở cả 10 cách


In [29]:
# 12.4. Input vừa/lớn: các cách nhanh khớp nhau (oracle O(n^2) không dùng được)
rng = random.Random(99)
for _ in range(20):
    n = rng.randint(1000, 5000)
    alphabet = rng.choice(["AB", "ABC", "ABCDEFGHIJKLMNOPQRSTUVWXYZ"])
    s = "".join(rng.choice(alphabet) for _ in range(n))
    k = rng.choice([0, 1, 5, n // 10, n // 2, n])
    a = replace_binary(s, k)
    b = replace_per_letter(s, k)
    c = replace_window_exact(s, k)
    d = replace_window_fast(s, k)
    assert a == b == c == d, (a, b, c, d, k)
print("12.4 OK: 20 test lớn khớp nhau")

12.4 OK: 20 test lớn khớp nhau


In [30]:
# 12.5. Chuỗi có cấu trúc: run dài + nhiễu, kiểm tra Cách 5 (nộp) với brute O(n^2) trên n = 1500
rng = random.Random(5)
for _ in range(5):
    s = "".join(rng.choice("AB") * rng.randint(1, 20) for _ in range(120))[:1500]
    k = rng.randint(0, 30)
    assert Solution().characterReplacement(s, k) == replace_brute(s, k)
print("12.5 OK")

12.5 OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên giảm bộ đếm khi dịch `l`

Dịch `l` mà không `cnt[s[l]] -= 1` làm bộ đếm chứa cả những ký tự **đã rời** cửa sổ ⇒ `mx` bị thổi phồng.

In [31]:
def wrong_no_decrement(s, k):
    cnt = [0] * 26
    l = mx = 0
    for r, ch in enumerate(s):
        i = ord(ch) - 65
        cnt[i] += 1
        mx = max(mx, cnt[i])
        if r - l + 1 - mx > k:
            l += 1                          # SAI: quên cnt[s[l]] -= 1
    return len(s) - l


rng = random.Random(1)
found = None
for _ in range(5000):
    s = "".join(rng.choice("ABC") for _ in range(rng.randint(1, 8)))
    k = rng.randint(0, 3)
    if wrong_no_decrement(s, k) != replace_window_fast(s, k):
        found = (s, k)
        break
print("phản ví dụ:", found, "| sai:", wrong_no_decrement(*found), "| đúng:", replace_window_fast(*found))

phản ví dụ: ('CACBBCA', 0) | sai: 3 | đúng: 2


**Vì sao sai?** `cnt` đếm cả ký tự đã bị bỏ, nên `mx` lớn hơn thực tế và điều kiện `size - mx > k` bị nới quá ⇒ cửa sổ to hơn mức cho phép.

### Lỗi 2 — So sánh `>= k` thay vì `> k`

In [32]:
def wrong_ge(s, k):
    cnt = [0] * 26
    l = mx = 0
    for r, ch in enumerate(s):
        i = ord(ch) - 65
        cnt[i] += 1
        mx = max(mx, cnt[i])
        if r - l + 1 - mx >= k:             # SAI: được đổi TỐI ĐA k lần, nên chỉ vi phạm khi > k
            cnt[ord(s[l]) - 65] -= 1
            l += 1
    return len(s) - l


print("ABAB, k=2 ->", "sai", wrong_ge("ABAB", 2), "| đúng", replace_window_fast("ABAB", 2))
rng = random.Random(2)
found = None
for _ in range(5000):
    s = "".join(rng.choice("AB") for _ in range(rng.randint(1, 8)))
    k = rng.randint(0, 4)
    if wrong_ge(s, k) != replace_window_fast(s, k):
        found = (s, k)
        break
print("một phản ví dụ khác:", found, "| sai:", wrong_ge(*found), "| đúng:", replace_window_fast(*found))

ABAB, k=2 -> sai 3 | đúng 4
một phản ví dụ khác: ('A', 0) | sai: 0 | đúng: 1


**Vì sao sai?** Đề cho phép **tối đa `k`** phép đổi, tức `len - mx == k` vẫn hợp lệ. Dùng `>= k` coi trường hợp đúng `k` là không hợp lệ ⇒ đáp án nhỏ đi.

### Lỗi 3 — Dùng `max` của cả chuỗi thay vì của cửa sổ

In [33]:
def wrong_global_max(s, k):
    mx_global = max(Counter(s).values())                  # SAI: max trên cả chuỗi
    l = 0
    for r in range(len(s)):
        if r - l + 1 - mx_global > k:
            l += 1
    return len(s) - l


rng = random.Random(3)
found = None
for _ in range(5000):
    s = "".join(rng.choice("AB") for _ in range(rng.randint(2, 9)))
    k = rng.randint(0, 3)
    if wrong_global_max(s, k) != replace_window_fast(s, k):
        found = (s, k)
        break
print("phản ví dụ:", found, "| sai:", wrong_global_max(*found), "| đúng:", replace_window_fast(*found))

phản ví dụ: ('ABABABAA', 2) | sai: 7 | đúng: 6


**Vì sao sai?** Chữ phổ biến nhất của **cả chuỗi** có thể nằm rải rác, không liên tiếp, nên không liên quan tới đoạn đang xét. `mx` phải là tần suất trong **cửa sổ** (hoặc kỷ lục các cửa sổ như Cách 5).

### Lỗi 4 — "Công thức tắt" `min(n, maxfreq + k)`

Ý nghĩ: "chữ phổ biến nhất xuất hiện `f` lần, đổi thêm `k` ký tự khác ⇒ `f + k`". Sai vì các ký tự phải **liên tiếp**.

In [34]:
def wrong_formula(s, k):
    return min(len(s), max(Counter(s).values()) + k)


print("ABABAB, k=1: sai =", wrong_formula("ABABAB", 1), "| đúng =", replace_window_fast("ABABAB", 1))
rng = random.Random(4)
for _ in range(5000):
    s = "".join(rng.choice("AB") for _ in range(rng.randint(2, 9)))
    k = rng.randint(0, 3)
    if wrong_formula(s, k) != replace_window_fast(s, k):
        print("một phản ví dụ khác:", (s, k), "| sai:", wrong_formula(s, k), "| đúng:", replace_window_fast(s, k))
        break

ABABAB, k=1: sai = 4 | đúng = 3
một phản ví dụ khác: ('BABBA', 0) | sai: 3 | đúng: 2


**Vì sao sai?** `ABABAB` có `A` ba lần nhưng đổi 1 ký tự chỉ nối được `ABA → AAA`: dài 3, không phải 4. Số lần xuất hiện **tổng** ≠ số lần xuất hiện **trong một đoạn liên tiếp**. Bài học: luôn thử công thức tắt với oracle trước khi tin.

## 14. Mẹo & nhận dạng pattern

- **"Đoạn con liên tiếp dài nhất thỏa điều kiện kiểu `tổng chi phí <= k`"** ⇒ cửa sổ trượt (hai con trỏ), đặc biệt khi điều kiện **đơn điệu** (đoạn con của đoạn hợp lệ cũng hợp lệ).
- **"Chi phí để đồng nhất = len - tần suất lớn nhất"** là mẹo quen thuộc cho các bài "đổi tối đa `k`" (cũng thấy ở Max Consecutive Ones III, Frequency of the Most Frequent Element...).
- **Đơn điệu ⇒ nhị phân đáp án** là phương án dự phòng khi chưa nghĩ ra cửa sổ trượt; kiểm tra khả thi cho một độ dài bằng cửa sổ cố định.
- **Bảng chữ cái cố định (26)** ⇒ mảng đếm thay `dict`, và `max(cnt)` coi như `O(1)`.
- **Mẹo `mx` không giảm:** khi chỉ cần độ dài lớn nhất, cửa sổ "không bao giờ co" và biến kỷ lục không cần cập nhật khi `l` dịch. Dùng `if` thay `while`.
- Khi nghi ngờ một tối ưu tinh vi, hãy **viết oracle định nghĩa** và để máy tìm phản ví dụ.

## 15. Biến thể

1. **Max Consecutive Ones III (LeetCode 1004):** mảng nhị phân, được đổi tối đa `k` số `0` thành `1`; tìm dãy `1` dài nhất. Chính là Cách 3 với chữ đích `1`.
2. **Chữ cái hoa lẫn thường / bảng chữ cái lớn:** thay mảng 26 bằng `Counter`/`dict`; Cách 5 vẫn đúng.
3. **Trả về chính đoạn con (hoặc vị trí bắt đầu):** Cách 4 cho `(l, r)` mỗi bước; theo dõi `best` kèm `l`.
4. **Chỉ được đổi thành một chữ cố định `c`:** đó là Cách 3 với `c` cố định.
5. **Đếm số đoạn con hợp lệ:** khi cửa sổ `[l, r]` hợp lệ (Cách 4), có `r - l + 1` đoạn con hợp lệ kết thúc tại `r` ⇒ cộng dồn.

In [35]:
# Biến thể 1: Max Consecutive Ones III
def longest_ones(nums, k):
    l = zeros = best = 0
    for r, x in enumerate(nums):
        if x == 0:
            zeros += 1
        while zeros > k:
            if nums[l] == 0:
                zeros -= 1
            l += 1
        best = max(best, r - l + 1)
    return best


def longest_ones_brute(nums, k):
    n = len(nums)
    return max((r - l for l in range(n) for r in range(l + 1, n + 1) if nums[l:r].count(0) <= k), default=0)


rng = random.Random(15)
for _ in range(500):
    nums = [rng.randint(0, 1) for _ in range(rng.randint(1, 12))]
    k = rng.randint(0, 4)
    assert longest_ones(nums, k) == longest_ones_brute(nums, k)
print("biến thể 1:", longest_ones([1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0], 2))        # 6


# Biến thể 5: đếm số đoạn con hợp lệ (có thể đồng nhất bằng <= k phép đổi)
def count_valid_substrings(s, k):
    cnt = [0] * 26
    l = total = 0
    for r, ch in enumerate(s):
        cnt[ord(ch) - 65] += 1
        while (r - l + 1) - max(cnt) > k:
            cnt[ord(s[l]) - 65] -= 1
            l += 1
        total += r - l + 1
    return total


def count_valid_brute(s, k):
    return sum(1 for l in range(len(s)) for r in range(l + 1, len(s) + 1)
               if need_by_definition(s[l:r]) <= k)


for _ in range(300):
    s = "".join(rng.choice("ABC") for _ in range(rng.randint(1, 10)))
    k = rng.randint(0, 3)
    assert count_valid_substrings(s, k) == count_valid_brute(s, k)
print("biến thể 5: số đoạn hợp lệ của AABABBA, k=1 =", count_valid_substrings("AABABBA", 1))

biến thể 1: 6
biến thể 5: số đoạn hợp lệ của AABABBA, k=1 = 20


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Mọi đoạn, đếm tăng dần | `O(n²)` | `O(1)` | Oracle `n` nhỏ |
| 2. Nhị phân độ dài | Đơn điệu + cửa sổ cố định | `O(26 n log n)` | `O(n)` | Khi chưa nghĩ ra cửa sổ trượt |
| 3. Từng chữ đích | ≤ `k` ký tự khác đích | `O(26 n)` | `O(1)` | Khi chữ đích bị cố định; liên hệ Max Consecutive Ones III |
| 4. Cửa sổ chuẩn | `while` + `max(cnt)` thật | `O(26 n)` | `O(1)` | Dễ chứng minh đúng |
| 5. Cửa sổ tối ưu | `mx` không giảm, `if` | `O(n)` | `O(1)` | **Khuyên dùng khi thi** |

Cả năm cách cho cùng đáp án trên mọi test ở mục 12.

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `ord("D") - ord("A")`; (b) `max([0, 3, 1])`; (c) `len("AABAB") - max(Counter("AABAB").values())`; (d) với `s = "AAAA"`, `k = 0` đáp án là bao nhiêu?

**Bài 2 (tính tay).** Với `s = "ABBB"`, `k = 2` và `s = "AABBC"`, `k = 1`, tính đáp án bằng cách liệt kê vài đoạn và dùng công thức `len - maxfreq`. Sau đó tính lại bằng Cách 5 theo từng bước `r`.

**Bài 3 (điền TODO).** Hoàn thành `char_replace_todo(s, k)` theo Cách 5.

In [36]:
def char_replace_todo(s, k):
    # TODO 1: cnt = [0] * 26 ; l = 0 ; mx = 0
    # TODO 2: for r, ch in enumerate(s): cnt[ord(ch) - 65] += 1
    # TODO 3: mx = max(mx, cnt[ord(ch) - 65])
    # TODO 4: nếu r - l + 1 - mx > k thì giảm cnt của s[l] và l += 1
    # TODO 5: trả về len(s) - l
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `char_replace_letter(s, k)` theo Cách 3 (với từng chữ đích) **không nhìn lại** mục 7. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** Viết `longest_window(s, k)` trả về cặp `(độ dài, vị trí bắt đầu)` của **một** đoạn dài nhất (nếu hòa, lấy vị trí bắt đầu nhỏ nhất). Kiểm tra: `("AABABBA", 1) → (4, 0)`; `("ABCDE", 0) → (1, 0)`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `3`; `3`; `5 - 3 = 2`; `4`. Bài 3: nhìn lại hàm `replace_window_fast` ở mục 7.
- **Mức 2 (vừa):** Bài 2: `ABBB, k=2` → cả chuỗi (`4 - 3 = 1 <= 2`) nên đáp án `4`; `AABBC, k=1` → các đoạn dài 4 như `AABB` hay `ABBC` đều cần 2 phép đổi (quá `k`), còn `AAB`, `ABB` cần 1 phép nên đáp án `3`. Bài 4: duyệt `c in set(s)`, đếm `bad`.
- **Mức 3 (sâu):** Bài 5: dùng Cách 4 (`while` và `max` thật) và cập nhật `best` kèm `l` khi `r - l + 1 > best`; chỉ cập nhật khi **lớn hơn hẳn** để giữ vị trí bắt đầu nhỏ nhất. Đối chiếu với oracle `replace_definition` về độ dài.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [37]:
# Bài 1
assert ord("D") - ord("A") == 3
assert max([0, 3, 1]) == 3
assert len("AABAB") - max(Counter("AABAB").values()) == 2
assert replace_definition("AAAA", 0) == 4

# Bài 2
assert replace_definition("ABBB", 2) == 4
assert replace_definition("AABBC", 1) == 3
assert replace_window_fast("ABBB", 2) == 4 and replace_window_fast("AABBC", 1) == 3


# Bài 3
def char_replace_todo_ans(s, k):
    cnt = [0] * 26
    l = mx = 0
    for r, ch in enumerate(s):
        cnt[ord(ch) - 65] += 1
        mx = max(mx, cnt[ord(ch) - 65])
        if r - l + 1 - mx > k:
            cnt[ord(s[l]) - 65] -= 1
            l += 1
    return len(s) - l


for s, k, expected in examples:
    assert char_replace_todo_ans(s, k) == expected


# Bài 4
def char_replace_letter(s, k):
    best = 0
    for c in set(s):
        l = bad = 0
        for r, ch in enumerate(s):
            bad += ch != c
            while bad > k:
                bad -= s[l] != c
                l += 1
            best = max(best, r - l + 1)
    return best


for s, k, expected in examples:
    assert char_replace_letter(s, k) == expected


# Bài 5
def longest_window(s, k):
    cnt = [0] * 26
    l = 0
    best, best_start = 0, 0
    for r, ch in enumerate(s):
        cnt[ord(ch) - 65] += 1
        while (r - l + 1) - max(cnt) > k:
            cnt[ord(s[l]) - 65] -= 1
            l += 1
        if r - l + 1 > best:
            best, best_start = r - l + 1, l
    return best, best_start


assert longest_window("AABABBA", 1) == (4, 0)
assert longest_window("ABCDE", 0) == (1, 0)
rng = random.Random(19)
for _ in range(300):
    s = "".join(rng.choice("ABC") for _ in range(rng.randint(1, 10)))
    k = rng.randint(0, len(s))
    length, start = longest_window(s, k)
    assert length == replace_definition(s, k)
    assert need_by_definition(s[start:start + length]) <= k
    assert char_replace_letter(s, k) == length == char_replace_todo_ans(s, k)
print("OK: toàn bộ đáp án đúng")

OK: toàn bộ đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template** (cửa sổ trượt, `mx` không giảm)

```python
class Solution:
    def characterReplacement(self, s: str, k: int) -> int:
        cnt = [0] * 26
        l = mx = 0
        for r, ch in enumerate(s):
            i = ord(ch) - ord("A")
            cnt[i] += 1
            mx = max(mx, cnt[i])
            if r - l + 1 - mx > k:                 # cửa sổ cần > k phép đổi
                cnt[ord(s[l]) - ord("A")] -= 1     # bỏ ký tự đầu
                l += 1
        return len(s) - l
```

**Checklist**

- [ ] Điều kiện vi phạm là `len - mx > k` (không phải `>= k`)?
- [ ] Khi dịch `l` đã **giảm** `cnt[s[l]]`?
- [ ] `mx` là tần suất trong cửa sổ (hoặc kỷ lục các cửa sổ), **không** phải của cả chuỗi?
- [ ] Nếu dùng `if` (Cách 5) thì trả `len(s) - l`; nếu dùng `while` thì theo dõi `best`?
- [ ] Không dùng công thức tắt `min(n, maxfreq + k)` (bỏ qua tính liên tiếp)?
- [ ] Đã test `k = 0`, `k = n`, một ký tự, chuỗi `ABABAB`, và so với oracle theo định nghĩa?